In [ ]:
pip install transformers

In [ ]:
pip install hf_transfer

In [ ]:
pip install accelerate

In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name ="Qwen/Qwen2.5-0.5B-Instruct"   
#"Qwen/Qwen2.5-7B-Instruct" -- avaliable in Hugging face
#"qwen3.5:0.8b" -- compresed by ollama
#"Qwen/Qwen2.5-0.5B-Instruct" -- in hugging face

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

C:\Users\KRESHAN88\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 290/290 [00:00<00:00, 5915.34it/s]


In [5]:
model.config._name_or_path

'Qwen/Qwen2.5-0.5B-Instruct'

In [6]:
tokenizer = AutoTokenizer.from_pretrained(model_name)


In [7]:
prompt = "Give me the capital of Srilanka?"
messages = [
    {"role": "system", "content": "You are a Tamil Ai Assistent"},
    {"role": "user", "content": prompt}
]


text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

print(text)

<|im_start|>system
You are a Tamil Ai Assistent<|im_end|>
<|im_start|>user
Give me the capital of Srilanka?<|im_end|>
<|im_start|>assistant



In [8]:
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)


In [9]:
model_inputs

{'input_ids': tensor([[151644,   8948,    198,   2610,    525,    264,  43783,  55986,   2688,
          18128, 151645,    198, 151644,    872,    198,  35127,    752,    279,
           6722,    315,    328,  30560,  26671,     30, 151645,    198, 151644,
          77091,    198]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1]])}

In [11]:
generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=512,
    temperature=0.7,   #  0 = original, 1 = more creative
    top_p=0.9,         # picks tokens from first 90%
    # top_k=50,        # picks from top 50 probable tokens
    do_sample=True     # must be True to use temperature, top_p, or top_k
)

In [ ]:
# generated_ids = model.generate(
#     **model_inputs,
#     max_new_tokens=512,
#     do_sample=False    # greedy decoding, always picks most probable token
# )

In [12]:
generated_ids

tensor([[151644,   8948,    198,   2610,    525,    264,  43783,  55986,   2688,
          18128, 151645,    198, 151644,    872,    198,  35127,    752,    279,
           6722,    315,    328,  30560,  26671,     30, 151645,    198, 151644,
          77091,    198,    785,   6722,    315,  33345,  42479,    374,   4254,
          80387,     13, 151645]])

In [13]:
generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

In [14]:
generated_ids

[tensor([   785,   6722,    315,  33345,  42479,    374,   4254,  80387,     13,
         151645])]

In [15]:
len([ 39814,      0,    362,   3460,   4128,   1614,    320,   4086,     44,
              8,    374,    264,    943,    315,  20443,  11229,   1614,   6188,
            311,   1882,    323,   6923,   3738,  12681,   1467,     13,   4220,
           4119,    525,  11136,   3118,    389,   5538,   6832,  12538,     11,
           7945,  42578,  77235,     11,    892,   2138,   1105,    311,   3535,
           2266,     11,   6923,  55787,  14507,     11,    323,   3705,    264,
           6884,   2088,    315,   5810,   4128,   9079,    382,   1592,  17452,
            315,    444,  10994,     82,   2924,   1447,     16,     13,   3070,
           1695,  95518,    444,  10994,     82,    525,   5990,   1602,   3460,
             11,   8482,  11728,    476,   1496,  32051,    315,   5029,     13,
           1096,   1379,   6147,   1105,    311,  12322,   6351,  12624,    304,
           4128,    821,    382,     17,     13,   3070,  36930,   2885,  95518,
           2379,    525,  16176,    389,  12767,  14713,    315,   1467,    821,
            504,    279,   7602,     11,   6467,     11,    323,   1008,   8173,
             11,  27362,   1105,    311,   3960,    264,   7205,   2088,    315,
          64667,  83789,    323,   6540,    382,     18,     13,   3070,  50359,
          95518,    444,  10994,     82,    646,    387,   1483,    369,   5257,
           5810,   4128,   8692,   9079,   1741,    438,   1467,   9755,     11,
          14468,     11,   3405,  35764,     11,  28285,   2022,     11,    323,
            803,    382,     19,     13,   3070,   3306,   7175,  95518,   3085,
          82687,   1075,   6236,  61905,    323,   7517,   1663,  13009,     11,
            444,  10994,     82,    646,  16579,    304,  20753,  21276,    448,
           3847,     11,   8241,  34549,    323,   2266,   1832,   9760,  14507,
            382,     20,     13,   3070,  65390,    938,  21144,    804,  95518,
           5976,   7988,     11,    444,  10994,     82,   1083,   4828,  30208,
          10520,   5435,    311,  15470,     11,  12345,     11,    323,    279,
           4650,  61751,    315,   7907,   2213,    382,  40381,    315,   1632,
          21309,   3460,   4128,   4119,   2924,   5085,    594,    425,   3399,
             11,   5264,  15469,    594,    479,   2828,   4013,     11,    323,
          54364,  14817,    594,   1207,  16948,     13,   4220,   4119,   3060,
            311,  37580,     11,  17461,    279,  22711,    315,   1128,  15235,
            646,  11075,    304,   5810,   4128,   8660,    323,   9471,     13,
         151645])

289

In [16]:
response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]


In [20]:
response

'The capital of Sri Lanka is Colombo.'